In [1]:
import boto3
import os
from pathlib import Path
from dotenv import load_dotenv
import sys

env_path = Path.cwd().parent.parent.joinpath("env").joinpath("dev.aws.env")

if env_path.exists:
    load_dotenv(dotenv_path=env_path)
    print("env loaded successfully. App Name = ", os.getenv("APP_NAME"))
else:
    print("Env file not found")

aws_url = os.getenv("AWS_URL")
region = os.getenv("REGION")
aws_access_key_id = os.getenv("AWS_ACCESS_KEY_ID")
aws_secret_access_key = os.getenv("AWS_SECRET_ACCESS_KEY")

env loaded successfully. App Name =  AWS PRACTICE MAC


In [2]:
snsClient = boto3.client('sns',
    endpoint_url= aws_url,
    region_name=region,
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key
)

snsClient

In [3]:
sqsClient = boto3.client('sqs',
    endpoint_url= aws_url,
    region_name=region,
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key
 )


In [4]:
# create new queue
response = sqsClient.create_queue(
    QueueName='MyLocalTestingQueue',
    Attributes={
        'DelaySeconds': '0',
        'MessageRetentionPeriod': '86400' # Retain for 1 day (in seconds)
    }
)

print(f"Queue Created successfully! URL: {response['QueueUrl']}")

Queue Created successfully! URL: http://localhost:4566/000000000000/MyLocalTestingQueue


In [5]:
# Receive message from queue
queue_url = "http://localhost:4566/000000000000/MyLocalTestingQueue"
message_from_queue =  sqsClient.receive_message(
    QueueUrl=queue_url
)

if message_from_queue.get("Messages") is  None:
    print("No messages received from queue or queue processed")
else:
    for msg in message_from_queue.get("Messages"):
        print(msg.get("Body"))

        # sqsClient.delete_message(
        #     QueueUrl=queue_url,
        #     ReceiptHandle=msg['ReceiptHandle']
        # )


{"Type": "Notification", "MessageId": "dac0b10b-7064-40a6-86b0-deff3131605f", "TopicArn": "arn:aws:sns:us-east-1:000000000000:Test topic for SNS", "Subject": "Order Status Update", "Message": "{\"order_id\": \"12345\", \"status\": \"shipped\"}", "Timestamp": "2026-06-29T10:53:49.000Z", "SignatureVersion": "1", "Signature": "FAKE", "SigningCertURL": "https://sns.us-east-1.amazonaws.com/SimpleNotificationService-fake.pem", "UnsubscribeURL": "http://localhost:4566/?Action=Unsubscribe&SubscriptionArn=arn:aws:sns:us-east-1:000000000000:example"}


In [6]:
snsClient.create_topic(Name="Test topic for SNS")

{'TopicArn': 'arn:aws:sns:us-east-1:000000000000:Test topic for SNS',
 'ResponseMetadata': {'RequestId': 'f5d26416-f0cb-422e-9ff6-588a2677336a',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'content-type': 'application/xml',
   'access-control-allow-origin': '*',
   'access-control-allow-methods': 'GET, POST, PUT, DELETE, HEAD, OPTIONS, PATCH',
   'access-control-allow-headers': '*',
   'access-control-expose-headers': '*',
   'x-amzn-requestid': '3079d71a-cbb4-49f7-841e-b958987deff0',
   'x-amz-request-id': '3079d71a-cbb4-49f7-841e-b958987deff0',
   'x-amz-id-2': 'bYm2qERRJCeiA+FIbnM0VH0A6TinYfI0vgGr7iLsJxbEgfhZKy0PJN7cqqr6E8JO',
   'content-length': '339',
   'date': 'Sat, 04 Jul 2026 06:42:05 GMT',
   'server': 'hypercorn-h11'},
  'RetryAttempts': 0}}

In [7]:
response = snsClient.create_topic(
    Name="my-strict-ordering-topic.fifo",
    Attributes={
        'FifoTopic': 'true',
        'ContentBasedDeduplication': 'true' # Optional auto-deduplication
    }
)

response

{'TopicArn': 'arn:aws:sns:us-east-1:000000000000:my-strict-ordering-topic.fifo',
 'ResponseMetadata': {'RequestId': '20f28f94-91d1-4ff9-b081-9d345e2d826f',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'content-type': 'application/xml',
   'access-control-allow-origin': '*',
   'access-control-allow-methods': 'GET, POST, PUT, DELETE, HEAD, OPTIONS, PATCH',
   'access-control-allow-headers': '*',
   'access-control-expose-headers': '*',
   'x-amzn-requestid': '48a05df7-b84b-491a-a3bc-1b7abfa9f4b1',
   'x-amz-request-id': '48a05df7-b84b-491a-a3bc-1b7abfa9f4b1',
   'x-amz-id-2': 'GBAoewlJ/2tzOlg8EZeB/+qOn/rKzKdE435sgHsqTyUEwNjNf74oLnNXQUz7KKUJ',
   'content-length': '350',
   'date': 'Sat, 04 Jul 2026 06:42:05 GMT',
   'server': 'hypercorn-h11'},
  'RetryAttempts': 0}}

In [8]:
# Subscribe an email address to the topic
subscription = snsClient.subscribe(
    TopicArn="arn:aws:sns:us-east-1:000000000000:Test topic for SNS",
    Protocol='sqs',
    Endpoint='arn:aws:sqs:us-east-1:000000000000:MyLocalTestingQueue'
)

print(f"Subscription ARN: {subscription['SubscriptionArn']}")
print("Note: Check your inbox to confirm the pending subscription confirmation.")


Subscription ARN: arn:aws:sns:us-east-1:000000000000:Test topic for SNS:2fbfb333-4641-413d-8c6f-01eacaaaa14a
Note: Check your inbox to confirm the pending subscription confirmation.


In [9]:
#publish new item to topic
publish_response = snsClient.publish(
    TopicArn="arn:aws:sns:us-east-1:000000000000:Test topic for SNS",
    Message='{"order_id": "12341", "status": "shipped"}',
    Subject='Order Status Update'
)

In [10]:
#read secret manager data
sm_client = boto3.client("secretsmanager",
    endpoint_url= aws_url,
    region_name=region,
    aws_access_key_id=aws_access_key_id,
                         aws_secret_access_key=aws_secret_access_key
 )